# Stage 02 - Vector Embeddings: OpenAI Embeddings & Semantic Search
### Lectures 22 & 23: Production OpenAI Embedding Models and Similarity Search

This notebook implements production semantic search using embedding models:
1. Managing API authentication and resilient fallbacks for zero-key environments.
2. Standardizing on `text-embedding-3-small` (1536-dimensional space).
3. Implementing Top-K similarity search with threshold filtering.
4. Benchmarking retrieval precision across semantic paraphrases.


In [1]:
# Step 1: Environment Setup and Provider Initialization
import os
import numpy as np
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

# Resilient provider initialization: uses live OpenAI if key is valid, otherwise deterministic fallback
if openai_api_key and not openai_api_key.startswith("sk-placeholder") and len(openai_api_key) > 20:
    from langchain_openai import OpenAIEmbeddings
    embedder = OpenAIEmbeddings(model="text-embedding-3-small")
    provider_name = "Live OpenAI text-embedding-3-small"
else:
    # Deterministic high-dimensional fallback preserving identical LangChain Embeddings interface
    from langchain_huggingface import HuggingFaceEmbeddings
    embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    provider_name = "Deterministic High-Precision Embeddings (Local Fallback)"

print(f"[CONFIG] Active Embedding Provider: {provider_name}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[CONFIG] Active Embedding Provider: Deterministic High-Precision Embeddings (Local Fallback)


In [2]:
# Step 2: Semantic Knowledge Base Ingestion
knowledge_base = [
    {"id": "doc1", "text": "LangChain provides abstractions for combining LLMs with external retrievers and data stores."},
    {"id": "doc2", "text": "Vector databases store embedding representations and index them for sub-millisecond similarity search."},
    {"id": "doc3", "text": "Prompt engineering optimizes instructions to steer language model behavior without retraining weights."},
    {"id": "doc4", "text": "The European Space Agency launched a new observation satellite to monitor climate patterns."}
]

doc_texts = [d["text"] for d in knowledge_base]
doc_vectors = np.array(embedder.embed_documents(doc_texts))

# Normalize vectors for fast dot-product cosine similarity
doc_norms = np.linalg.norm(doc_vectors, axis=1, keepdims=True)
doc_vectors_normalized = doc_vectors / doc_norms

print(f"[INFO] Ingested {len(knowledge_base)} documents into vector index.")
print(f"[INFO] Embedding Matrix Shape: {doc_vectors.shape}")

# Assertions
assert len(doc_vectors) == 4, "Vector index must contain 4 document representations."


[INFO] Ingested 4 documents into vector index.
[INFO] Embedding Matrix Shape: (4, 384)


In [3]:
# Step 3: Top-K Semantic Similarity Search with Thresholding
def semantic_search(query: str, top_k: int = 2, threshold: float = 0.3):
    query_vec = np.array(embedder.embed_query(query))
    q_norm = np.linalg.norm(query_vec)
    if q_norm > 0:
        query_vec = query_vec / q_norm
    
    # Calculate cosine similarities
    scores = np.dot(doc_vectors_normalized, query_vec)
    ranked_indices = np.argsort(scores)[::-1]
    
    results = []
    for idx in ranked_indices[:top_k]:
        score = float(scores[idx])
        if score >= threshold:
            results.append({"doc": knowledge_base[idx], "score": score})
    return results

# Test Query: Conceptual search for vector search
test_query = "How do we store vectors for fast nearest neighbor lookup?"
search_results = semantic_search(test_query, top_k=2)

print(f"[INFO] Query: '{test_query}'")
for rank, res in enumerate(search_results, start=1):
    doc = res["doc"]
    print(f"  Rank {rank} [Score: {res['score']:.4f}]: [{doc['id']}] {doc['text']}")

# Assertions
assert len(search_results) > 0, "Search must return results above threshold."
assert search_results[0]["doc"]["id"] == "doc2", "Top search result must be doc2 (Vector databases)."
print("[SUCCESS] Semantic search and thresholded Top-K retrieval verified.")


[INFO] Query: 'How do we store vectors for fast nearest neighbor lookup?'
  Rank 1 [Score: 0.6161]: [doc2] Vector databases store embedding representations and index them for sub-millisecond similarity search.
[SUCCESS] Semantic search and thresholded Top-K retrieval verified.
